# XGBoost Model Optimisation

This notebook keeps the same final comparison design used for the other tuned models:

1. Model settings are selected without using the final validation or test sets.
2. The probability threshold is selected only on the validation set.
3. The final test set is evaluated once after the model settings and threshold are fixed.
4. The validation recall target is 80%.

XGBoost needs one extra internal development split because it builds trees sequentially. That internal early-stopping set is used to choose the learning rate and number of boosting rounds. It is separate from the validation set used for threshold selection.


In [1]:
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

print("scikit-learn version:", sklearn.__version__)
print("XGBoost version:", xgb.__version__)


scikit-learn version: 1.9.0
XGBoost version: 3.2.0


## 1. Load the cleaned dataset and create the output folder


In [2]:
PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = (
        parent
        / "Processed_Dataset"
        / "diabetic_data_cleaned_stage1.csv"
    )

    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError(
        "Could not find "
        "Processed_Dataset/diabetic_data_cleaned_stage1.csv"
    )

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Model_Results"
    / "xgboost_optimisation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(DATA_PATH)

print("Dataset path:")
print(DATA_PATH)

print("\nDataset shape:")
print(df.shape)

df.head()


Dataset path:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Processed_Dataset/diabetic_data_cleaned_stage1.csv

Dataset shape:
(69987, 56)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,diabetesMed,readmitted,readmitted_30,hba1c_group,primary_diagnosis,age_group,discharge_group,race_group,admission_source_group,medical_specialty_group
0,24437208,135,Caucasian,Female,[50-60),2,1,1,8,Cardiology,...,Yes,<30,1,No test was performed,Circulatory,30-60,Home,Caucasian,Physician/clinic referral,Cardiology
1,29758806,378,Caucasian,Female,[50-60),3,1,1,2,Surgery-Neuro,...,No,NO,0,No test was performed,Musculoskeletal,30-60,Home,Caucasian,Physician/clinic referral,Surgery
2,189899286,729,Caucasian,Female,[80-90),1,3,7,4,InternalMedicine,...,Yes,NO,0,Normal result of the test,Injury,>60,Other,Caucasian,Emergency room,Internal Medicine
3,64331490,774,Caucasian,Female,[80-90),1,1,7,3,InternalMedicine,...,Yes,NO,0,"High, medication changed",Other,>60,Home,Caucasian,Emergency room,Internal Medicine
4,14824206,927,AfricanAmerican,Female,[30-40),1,1,7,5,InternalMedicine,...,Yes,NO,0,No test was performed,Genitourinary,30-60,Home,AfricanAmerican,Emergency room,Internal Medicine


## 2. Select the same modelling features used by the other tuned models


In [3]:
target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed"
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

model_features = (
    categorical_features
    + numeric_features
)

missing_features = [
    feature
    for feature in model_features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "These modelling features are missing: "
        f"{missing_features}"
    )

X = df[model_features].copy()
y = df[target_col].astype(int).copy()

print("X shape:")
print(X.shape)

print("\nFeatures used:")
print(X.columns.tolist())

print("\nTarget counts:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))


X shape:
(69987, 18)

Features used:
['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Target counts:
readmitted_30
0    63702
1     6285
Name: count, dtype: int64

Target proportions:
readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64


In [4]:
forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
}

unexpected_features = (
    forbidden_features
    .intersection(X.columns)
)

assert not unexpected_features, (
    "Unexpected or potentially leaking features found: "
    f"{unexpected_features}"
)

assert not X.columns.duplicated().any(), (
    "Duplicate column names were found in X."
)

assert len(X) == len(y), (
    "X and y contain different numbers of rows."
)

assert y.isna().sum() == 0, (
    "The target contains missing values."
)

assert set(y.unique()).issubset({0, 1}), (
    "The target must contain only 0 and 1."
)

print("Feature and target checks passed.")


Feature and target checks passed.


## 3. Create development, validation, test, and early-stopping sets

The final model-training, validation, and test proportions remain 64%, 16%, and 20% of the full dataset.

XGBoost also needs an internal early-stopping set. It is taken from the 64% model-training portion and is used only while choosing the learning rate and number of boosting rounds. After those choices are fixed, the final XGBoost model is refitted on the complete 64% model-training set.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_model_train, X_val, y_model_train, y_val = (
    train_test_split(
        X_train,
        y_train,
        test_size=0.25,
        stratify=y_train,
        random_state=42
    )
)

X_search_train, X_early_stop, y_search_train, y_early_stop = (
    train_test_split(
        X_model_train,
        y_model_train,
        test_size=0.20,
        stratify=y_model_train,
        random_state=42
    )
)

split_summary = pd.DataFrame({
    "split": [
        "CV search training",
        "Internal early stopping",
        "Full model training after selection",
        "Threshold validation",
        "Final test"
    ],
    "rows": [
        len(X_search_train),
        len(X_early_stop),
        len(X_model_train),
        len(X_val),
        len(X_test)
    ],
    "positive_rate": [
        y_search_train.mean(),
        y_early_stop.mean(),
        y_model_train.mean(),
        y_val.mean(),
        y_test.mean()
    ]
})

split_summary


,split,rows,positive_rate
0,CV search training,33592,0.089813
1,Internal early stopping,8399,0.089773
2,Full model training after selection,41991,0.089805
3,Threshold validation,13998,0.089799
4,Final test,13998,0.089799


## 4. Preprocessing

XGBoost does not need standardisation. Categorical variables are imputed and one-hot encoded. Numeric variables are median-imputed. The same feature preparation is used for the cross-validation search, early stopping, validation threshold selection, and final test evaluation.


In [6]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

xgboost_preprocess = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
        (
            "numeric",
            numeric_transformer,
            numeric_features
        )
    ],
    remainder="drop"
)

print("XGBoost preprocessing created.")


XGBoost preprocessing created.


## 5. Evaluation and threshold-selection helper functions


In [7]:
def evaluate_predictions_from_proba(
    y_true,
    y_proba,
    threshold=0.5,
    model_name="Model"
):
    """Evaluate binary predictions created from probabilities."""

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    y_pred = (
        y_proba >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total = tn + fp + fn + tp
    actual_positive = tp + fn
    actual_negative = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    specificity = (
        tn / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_positive_rate = (
        fp / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_negative_rate = (
        fn / actual_positive
        if actual_positive > 0
        else np.nan
    )

    predicted_positive_rate = (
        predicted_positive / total
        if total > 0
        else np.nan
    )

    patients_flagged_per_true_readmission = (
        predicted_positive / tp
        if tp > 0
        else np.nan
    )

    return {
        "model": model_name,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f2": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            zero_division=0
        ),
        "auroc": roc_auc_score(
            y_true,
            y_proba
        ),
        "auprc": average_precision_score(
            y_true,
            y_proba
        ),
        "brier_score": brier_score_loss(
            y_true,
            y_proba
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "predicted_positive": int(predicted_positive),
        "predicted_negative": int(predicted_negative),
        "predicted_positive_rate": predicted_positive_rate,
        "patients_flagged_per_true_readmission_found": (
            patients_flagged_per_true_readmission
        )
    }


In [8]:
def confusion_matrix_from_proba(
    y_true,
    y_proba,
    threshold=0.5
):
    """Create a labelled confusion matrix from probabilities."""

    y_pred = (
        np.asarray(y_proba) >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    return pd.DataFrame(
        cm,
        index=[
            "Actual not readmitted",
            "Actual readmitted"
        ],
        columns=[
            "Predicted not readmitted",
            "Predicted readmitted"
        ]
    )


In [9]:
def threshold_sweep(
    y_true,
    y_proba,
    model_name="Model",
    thresholds=None
):
    """Calculate performance across probability thresholds."""

    if thresholds is None:
        thresholds = np.round(
            np.arange(
                0.01,
                0.951,
                0.01
            ),
            2
        )

    results = [
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=threshold,
            model_name=model_name
        )
        for threshold in thresholds
    ]

    return pd.DataFrame(results)


In [10]:
def choose_threshold_for_minimum_recall(
    y_true,
    y_proba,
    min_recall=0.80,
    model_name="Model"
):
    """
    Select the threshold with the lowest false-positive rate
    among thresholds that achieve the required recall.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    false_positive_rates, recalls, thresholds = roc_curve(
        y_true,
        y_proba,
        drop_intermediate=False
    )

    candidate_table = pd.DataFrame({
        "threshold": thresholds,
        "recall": recalls,
        "false_positive_rate": false_positive_rates,
        "specificity": 1 - false_positive_rates
    })

    candidate_table = candidate_table[
        np.isfinite(
            candidate_table["threshold"]
        )
    ].copy()

    eligible_candidates = candidate_table[
        candidate_table["recall"] >= min_recall
    ].copy()

    if eligible_candidates.empty:
        raise ValueError(
            "No threshold achieved recall >= "
            f"{min_recall:.2f}."
        )

    eligible_candidates = (
        eligible_candidates
        .sort_values(
            by=[
                "false_positive_rate",
                "threshold"
            ],
            ascending=[
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )

    selected_threshold = float(
        eligible_candidates.iloc[0]["threshold"]
    )

    selected_metrics = evaluate_predictions_from_proba(
        y_true=y_true,
        y_proba=y_proba,
        threshold=selected_threshold,
        model_name=model_name
    )

    return (
        selected_threshold,
        selected_metrics,
        eligible_candidates
    )


## 6. Stage 1: tune tree structure, sampling, regularisation, and class weighting

The first search keeps the learning rate and number of trees fixed. It asks a focused question: what kind of weak tree should XGBoost add at each boosting step?

AUPRC is the main search score because only about 9% of encounters are readmitted. AUROC is also saved for comparison. Recall at threshold 0.5 is not used to choose hyperparameters because threshold selection is a later, separate step.


In [11]:
RECALL_TARGET = 0.80
STRUCTURE_SEARCH_ITERATIONS = 30

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

negative_count = int((y_search_train == 0).sum())
positive_count = int((y_search_train == 1).sum())
imbalance_ratio = negative_count / positive_count

print(
    f"Required validation recall: "
    f"{RECALL_TARGET:.0%}"
)
print("Cross-validation folds:", cross_validation.n_splits)
print("Structure search candidates:", STRUCTURE_SEARCH_ITERATIONS)
print("Negative to positive ratio:", imbalance_ratio)


Required validation recall: 80%
Cross-validation folds: 5
Structure search candidates: 30
Negative to positive ratio: 10.134239310573417


In [12]:
xgboost_structure_param_distributions = {
    "model__max_depth": [
        2,
        3,
        4,
        5,
        6
    ],

    "model__min_child_weight": [
        1,
        2,
        5,
        10,
        20
    ],

    "model__gamma": [
        0.0,
        0.05,
        0.10,
        0.25,
        0.50,
        1.00
    ],

    "model__subsample": [
        0.60,
        0.75,
        0.90,
        1.00
    ],

    "model__colsample_bytree": [
        0.50,
        0.70,
        0.85,
        1.00
    ],

    "model__reg_alpha": [
        0.0,
        0.01,
        0.10,
        0.50,
        1.00,
        5.00
    ],

    "model__reg_lambda": [
        0.50,
        1.00,
        2.00,
        5.00,
        10.00,
        20.00
    ],

    "model__scale_pos_weight": [
        1.0,
        2.0,
        4.0,
        6.0,
        float(imbalance_ratio)
    ],

    "model__max_delta_step": [
        0,
        1,
        3
    ]
}


In [13]:
xgboost_search_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            xgboost_preprocess
        ),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                n_estimators=400,
                learning_rate=0.05,
                tree_method="hist",
                importance_type="gain",
                random_state=42,
                n_jobs=1
            )
        )
    ]
)

xgboost_search_pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default

In [14]:
xgboost_structure_search = RandomizedSearchCV(
    estimator=xgboost_search_pipeline,

    param_distributions=(
        xgboost_structure_param_distributions
    ),

    n_iter=STRUCTURE_SEARCH_ITERATIONS,

    scoring={
        "auprc": "average_precision",
        "auroc": "roc_auc"
    },

    refit="auprc",

    cv=cross_validation,

    n_jobs=-1,

    verbose=1,

    random_state=42,

    return_train_score=True,

    error_score="raise"
)

xgboost_structure_search.fit(
    X_search_train,
    y_search_train
)


Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__colsample_bytree': [0.5, 0.7, ...], 'model__gamma': [0.0, 0.05, ...], 'model__max_delta_step': [0, 1, ...], 'model__max_depth': [2, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","{'auprc': 'average_precision', 'auroc': 'roc_auc'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'auprc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These split

## 7. Inspect the strongest structure candidates


In [15]:
print("Best XGBoost structure parameters:")

for parameter, value in (
    xgboost_structure_search
    .best_params_
    .items()
):
    print(f"{parameter}: {value}")

print("\nBest cross-validation AUPRC:")
print(xgboost_structure_search.best_score_)


Best XGBoost structure parameters:
model__subsample: 0.6
model__scale_pos_weight: 6.0
model__reg_lambda: 20.0
model__reg_alpha: 1.0
model__min_child_weight: 10
model__max_depth: 2
model__max_delta_step: 1
model__gamma: 1.0
model__colsample_bytree: 1.0

Best cross-validation AUPRC:
0.15083828361630439


In [16]:
xgboost_structure_cv_results = pd.DataFrame(
    xgboost_structure_search.cv_results_
)

xgboost_structure_cv_results_selected = (
    xgboost_structure_cv_results[
        [
            "rank_test_auprc",
            "mean_test_auprc",
            "std_test_auprc",
            "mean_train_auprc",
            "std_train_auprc",
            "mean_test_auroc",
            "std_test_auroc",
            "param_model__max_depth",
            "param_model__min_child_weight",
            "param_model__gamma",
            "param_model__subsample",
            "param_model__colsample_bytree",
            "param_model__reg_alpha",
            "param_model__reg_lambda",
            "param_model__scale_pos_weight",
            "param_model__max_delta_step"
        ]
    ]
    .sort_values("rank_test_auprc")
    .reset_index(drop=True)
)

xgboost_structure_cv_results_selected[
    "train_validation_auprc_gap"
] = (
    xgboost_structure_cv_results_selected[
        "mean_train_auprc"
    ]
    -
    xgboost_structure_cv_results_selected[
        "mean_test_auprc"
    ]
)

xgboost_structure_cv_results_selected.to_csv(
    OUTPUT_DIR
    / "xgboost_structure_cv_results.csv",
    index=False
)

xgboost_structure_cv_results_selected.head(20)


,rank_test_auprc,mean_test_auprc,std_test_auprc,mean_train_auprc,std_train_auprc,mean_test_auroc,std_test_auroc,param_model__max_depth,param_model__min_child_weight,param_model__gamma,param_model__subsample,param_model__colsample_bytree,param_model__reg_alpha,param_model__reg_lambda,param_model__scale_pos_weight,param_model__max_delta_step,train_validation_auprc_gap
0,1,0.150838,0.008866,0.172018,0.003035,0.629633,0.011662,2,10,1.00,0.60,1.00,1.00,20.0,6.000000,1,0.021180
1,2,0.150205,0.009185,0.171990,0.003630,0.629886,0.011427,2,1,0.25,0.60,0.70,0.01,20.0,6.000000,0,0.021785
2,3,0.150128,0.008792,0.174063,0.004172,0.628636,0.011950,2,20,0.00,0.75,0.70,0.01,5.0,4.000000,0,0.023935
3,4,0.149969,0.009092,0.171399,0.003893,0.630023,0.011753,2,10,0.10,0.60,0.50,0.00,20.0,4.000000,0,0.021430
4,5,0.149318,0.009218,0.174180,0.003738,0.627795,0.012527,2,5,0.25,0.90,1.00,5.00,1.0,6.000000,0,0.024862
5,6,0.148967,0.009513,0.210487,0.003540,0.628226,0.010407,3,5,0.00,0.75,0.70,0.10,10.0,2.000000,0,0.061520
6,7,0.148948,0.008562,0.198748,0.004393,0.625490,0.011775,3,20,0.50,0.90,0.50,5.00,10.0,10.134239,3,0.049800
7,8,0.148792,0.009524,0.211525,0.003873,0.626037,0.011194,3,5,0.10,0.90,1.00,0.50,5.0,1.000000,1,0.062733
8,9,0.148474,0.009793,0.212058,0.004550,0.625517,0.011921,3,5,1.00,0.75,1.00,1.00,10.0,10.134239,0,0.063583
9,10,0.147897,0.009201,0.207677,0.003497,0.627841,0.010472,3,10,0.10,0.60,0.50,0.01,2.0,6.000000,1,0.059780


In [17]:
selected_structure_params = {
    parameter.replace("model__", ""): value
    for parameter, value in (
        xgboost_structure_search
        .best_params_
        .items()
    )
}

selected_structure_params


{'subsample': 0.6,
 'scale_pos_weight': 6.0,
 'reg_lambda': 20.0,
 'reg_alpha': 1.0,
 'min_child_weight': 10,
 'max_depth': 2,
 'max_delta_step': 1,
 'gamma': 1.0,
 'colsample_bytree': 1.0}

## 8. Stage 2: choose the learning rate and number of boosting rounds with early stopping

Random Forest trees are independent, but XGBoost trees are added one after another. The learning rate controls how much each new tree can change the model. A smaller learning rate normally needs more trees.

Each learning-rate candidate is trained with a deliberately large maximum of 2,000 trees. Training stops when internal early-stopping AUPRC has not improved for 75 rounds. The validation set used later for threshold selection remains untouched.


In [18]:
early_stop_preprocess = clone(xgboost_preprocess)

X_search_transformed = (
    early_stop_preprocess
    .fit_transform(X_search_train)
)

X_early_stop_transformed = (
    early_stop_preprocess
    .transform(X_early_stop)
)

print("Transformed search-training shape:")
print(X_search_transformed.shape)

print("\nTransformed early-stopping shape:")
print(X_early_stop_transformed.shape)


Transformed search-training shape:
(33592, 46)

Transformed early-stopping shape:
(8399, 46)


In [19]:
LEARNING_RATE_CANDIDATES = [
    0.02,
    0.03,
    0.05,
    0.08,
    0.10
]

MAX_BOOSTING_ROUNDS = 2000
EARLY_STOPPING_ROUNDS = 75

learning_rate_results = []
early_stopped_models = {}

for learning_rate in LEARNING_RATE_CANDIDATES:
    candidate_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        n_estimators=MAX_BOOSTING_ROUNDS,
        learning_rate=learning_rate,
        tree_method="hist",
        importance_type="gain",
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        **selected_structure_params
    )

    candidate_model.fit(
        X_search_transformed,
        y_search_train,
        eval_set=[
            (
                X_early_stop_transformed,
                y_early_stop
            )
        ],
        verbose=False
    )

    early_stop_proba = (
        candidate_model
        .predict_proba(
            X_early_stop_transformed
        )[:, 1]
    )

    best_iteration = int(
        candidate_model.best_iteration
    )

    learning_rate_results.append({
        "learning_rate": learning_rate,
        "selected_n_estimators": best_iteration + 1,
        "early_stop_auprc": average_precision_score(
            y_early_stop,
            early_stop_proba
        ),
        "early_stop_auroc": roc_auc_score(
            y_early_stop,
            early_stop_proba
        ),
        "early_stop_brier_score": brier_score_loss(
            y_early_stop,
            early_stop_proba
        ),
        "xgboost_best_score": float(
            candidate_model.best_score
        )
    })

    early_stopped_models[
        learning_rate
    ] = candidate_model

learning_rate_results = (
    pd.DataFrame(learning_rate_results)
    .sort_values(
        by=[
            "early_stop_auprc",
            "early_stop_auroc",
            "selected_n_estimators"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

learning_rate_results.to_csv(
    OUTPUT_DIR
    / "xgboost_learning_rate_early_stopping_results.csv",
    index=False
)

learning_rate_results


,learning_rate,selected_n_estimators,early_stop_auprc,early_stop_auroc,early_stop_brier_score,xgboost_best_score
0,0.10,196,0.161170,0.647880,0.152226,0.160095
1,0.08,187,0.160827,0.647463,0.152184,0.160129
2,0.05,194,0.158459,0.646367,0.153088,0.157402
3,0.02,634,0.157447,0.645969,0.153189,0.156415
4,0.03,49,0.155131,0.639003,0.155369,0.156181


In [20]:
selected_learning_rate = float(
    learning_rate_results.iloc[0][
        "learning_rate"
    ]
)

selected_n_estimators = int(
    learning_rate_results.iloc[0][
        "selected_n_estimators"
    ]
)

print("Selected learning rate:")
print(selected_learning_rate)

print("\nSelected number of boosting rounds:")
print(selected_n_estimators)


Selected learning rate:
0.1

Selected number of boosting rounds:
196


## 9. Refit the selected XGBoost model on the complete model-training set

The early-stopping set now returns to the training data. The final selected model is fitted on the full 64% model-training set using the chosen structure, learning rate, and number of boosting rounds. The threshold-validation and final-test sets are still untouched.


In [21]:
final_xgboost_params = deepcopy(
    selected_structure_params
)

final_xgboost_params.update({
    "learning_rate": selected_learning_rate,
    "n_estimators": selected_n_estimators
})

best_xgboost_model = Pipeline(
    steps=[
        (
            "preprocess",
            clone(xgboost_preprocess)
        ),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                importance_type="gain",
                random_state=42,
                n_jobs=-1,
                **final_xgboost_params
            )
        )
    ]
)

best_xgboost_model.fit(
    X_model_train,
    y_model_train
)

best_xgboost_model


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['gender','race_group','age_group',...,'number_emergency', 'number_inpatient','number_diagnoses']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifyi

In [22]:
fitted_xgboost = (
    best_xgboost_model
    .named_steps["model"]
)

xgboost_structure_summary = pd.Series({
    "number_of_boosting_rounds": int(
        fitted_xgboost.n_estimators
    ),
    "learning_rate": float(
        fitted_xgboost.learning_rate
    ),
    "max_depth": fitted_xgboost.max_depth,
    "min_child_weight": fitted_xgboost.min_child_weight,
    "gamma": fitted_xgboost.gamma,
    "subsample": fitted_xgboost.subsample,
    "colsample_bytree": fitted_xgboost.colsample_bytree,
    "reg_alpha": fitted_xgboost.reg_alpha,
    "reg_lambda": fitted_xgboost.reg_lambda,
    "scale_pos_weight": fitted_xgboost.scale_pos_weight,
    "max_delta_step": fitted_xgboost.max_delta_step
})

xgboost_structure_summary.to_csv(
    OUTPUT_DIR
    / "xgboost_selected_structure_summary.csv",
    header=["value"]
)

xgboost_structure_summary


number_of_boosting_rounds    196.0
learning_rate                  0.1
max_depth                      2.0
min_child_weight              10.0
gamma                          1.0
subsample                      0.6
colsample_bytree               1.0
reg_alpha                      1.0
reg_lambda                    20.0
scale_pos_weight               6.0
max_delta_step                 1.0
dtype: float64

## 10. Evaluate the selected model on the validation set at the default threshold


In [23]:
y_val_proba_xgboost = (
    best_xgboost_model
    .predict_proba(X_val)[:, 1]
)

print(
    "Minimum validation probability:",
    y_val_proba_xgboost.min()
)

print(
    "Maximum validation probability:",
    y_val_proba_xgboost.max()
)

print(
    "Number of unique validation probabilities:",
    np.unique(y_val_proba_xgboost).size
)


Minimum validation probability: 0.10168776
Maximum validation probability: 0.8235653
Number of unique validation probabilities: 13978


In [24]:
xgboost_probability_summary = (
    pd.DataFrame({
        "actual_class": np.asarray(y_val),
        "predicted_readmission_probability": (
            y_val_proba_xgboost
        )
    })
    .groupby("actual_class")
    ["predicted_readmission_probability"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

xgboost_probability_summary.to_csv(
    OUTPUT_DIR
    / "xgboost_validation_probability_summary.csv"
)

xgboost_probability_summary


,count,mean,std,min,10%,25%,50%,75%,90%,max
actual_class,,,,,,,,,,
0,12741.0,0.348942,0.110893,0.101688,0.217908,0.263203,0.330532,0.431257,0.496809,0.823565
1,1257.0,0.401345,0.117671,0.145870,0.254744,0.306402,0.399770,0.480135,0.544411,0.773899


In [25]:
xgboost_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_xgboost,
        threshold=0.5,
        model_name=(
            "XGBoost validation default"
        )
    )
)

pd.Series(
    xgboost_val_default_results
)


model                                          XGBoost validation default
threshold                                                             0.5
accuracy                                                          0.84212
precision                                                        0.171606
recall                                                           0.198091
specificity                                                      0.905659
false_positive_rate                                              0.094341
false_negative_rate                                              0.801909
f1                                                                 0.1839
f2                                                               0.192159
auroc                                                            0.628768
auprc                                                            0.140877
brier_score                                                      0.155444
true_negative                         

## 11. Select the validation threshold that reaches at least 80% recall

Among all validation thresholds that achieve the recall target, the rule selects the threshold with the lowest false-positive rate. This makes the model comparison fair because the same operating-point rule was used for the other tuned models.


In [26]:
(
    xgboost_selected_threshold,
    xgboost_val_selected_results,
    xgboost_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_xgboost,
    min_recall=RECALL_TARGET,
    model_name=(
        "XGBoost validation selected"
    )
)

print("Selected XGBoost threshold:")
print(xgboost_selected_threshold)

pd.DataFrame([
    xgboost_val_default_results,
    xgboost_val_selected_results
])


Selected XGBoost threshold:
0.2904823422431946


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,XGBoost validation default,0.500000,0.842120,0.171606,0.198091,0.905659,0.094341,0.801909,0.18390,0.192159,...,0.140877,0.155444,11539,1202,1008,249,1451,12547,0.103658,5.827309
1,XGBoost validation selected,0.290482,0.397914,0.109550,0.800318,0.358214,0.641786,0.199682,0.19272,0.353951,...,0.140877,0.155444,4564,8177,251,1006,9183,4815,0.656022,9.128231


In [27]:
xgboost_eligible_thresholds.to_csv(
    OUTPUT_DIR
    / "xgboost_eligible_validation_thresholds.csv",
    index=False
)

xgboost_eligible_thresholds.head(20)


,threshold,recall,false_positive_rate,specificity
0,0.290482,0.800318,0.641786,0.358214
1,0.290458,0.800318,0.641865,0.358135
2,0.290449,0.800318,0.641943,0.358057
3,0.290425,0.800318,0.642022,0.357978
4,0.290318,0.801114,0.642022,0.357978
5,0.290310,0.801114,0.642100,0.357900
6,0.290305,0.801114,0.642179,0.357821
7,0.290291,0.801114,0.642257,0.357743
8,0.290279,0.801114,0.642336,0.357664
9,0.290258,0.801114,0.642414,0.357586


In [28]:
xgboost_threshold_sweep = threshold_sweep(
    y_true=y_val,
    y_proba=y_val_proba_xgboost,
    model_name="XGBoost validation"
)

xgboost_threshold_sweep.to_csv(
    OUTPUT_DIR
    / "xgboost_validation_threshold_sweep.csv",
    index=False
)

xgboost_threshold_sweep[
    [
        "threshold",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f1",
        "f2",
        "true_positive",
        "false_positive",
        "false_negative",
        "predicted_positive_rate"
    ]
]


,threshold,recall,precision,specificity,false_positive_rate,f1,f2,true_positive,false_positive,false_negative,predicted_positive_rate
0,0.01,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
1,0.02,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
2,0.03,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
3,0.04,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
4,0.05,1.0,0.089799,0.0,1.0,0.164798,0.330337,1257,12741,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
90,0.91,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
91,0.92,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
92,0.93,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0
93,0.94,0.0,0.000000,1.0,0.0,0.000000,0.000000,0,0,1257,0.0


In [29]:
comparison_columns = [
    "model",
    "threshold",
    "auprc",
    "auroc",
    "brier_score",
    "accuracy",
    "recall",
    "precision",
    "specificity",
    "false_positive_rate",
    "false_negative_rate",
    "f1",
    "f2",
    "true_positive",
    "true_negative",
    "false_positive",
    "false_negative",
    "predicted_positive_rate",
    "patients_flagged_per_true_readmission_found"
]

xgboost_validation_comparison = pd.DataFrame([
    xgboost_val_default_results,
    xgboost_val_selected_results
])

xgboost_validation_comparison = (
    xgboost_validation_comparison[
        comparison_columns
    ]
)

xgboost_validation_comparison.to_csv(
    OUTPUT_DIR
    / "xgboost_validation_results.csv",
    index=False
)

xgboost_validation_comparison


,model,threshold,auprc,auroc,brier_score,accuracy,recall,precision,specificity,false_positive_rate,false_negative_rate,f1,f2,true_positive,true_negative,false_positive,false_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,XGBoost validation default,0.500000,0.140877,0.628768,0.155444,0.842120,0.198091,0.171606,0.905659,0.094341,0.801909,0.18390,0.192159,249,11539,1202,1008,0.103658,5.827309
1,XGBoost validation selected,0.290482,0.140877,0.628768,0.155444,0.397914,0.800318,0.109550,0.358214,0.641786,0.199682,0.19272,0.353951,1006,4564,8177,251,0.656022,9.128231


In [30]:
xgboost_val_selected_cm = (
    confusion_matrix_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_xgboost,
        threshold=(
            xgboost_selected_threshold
        )
    )
)

xgboost_val_selected_cm.to_csv(
    OUTPUT_DIR
    / "xgboost_validation_confusion_matrix.csv"
)

xgboost_val_selected_cm


,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4564,8177
Actual readmitted,251,1006


In [31]:
y_val_pred_xgboost = (
    y_val_proba_xgboost
    >= xgboost_selected_threshold
).astype(int)

print(
    classification_report(
        y_val,
        y_val_pred_xgboost,
        target_names=[
            "Not readmitted",
            "Readmitted"
        ],
        zero_division=0
    )
)


                precision    recall  f1-score   support

Not readmitted       0.95      0.36      0.52     12741
    Readmitted       0.11      0.80      0.19      1257

      accuracy                           0.40     13998
     macro avg       0.53      0.58      0.36     13998
  weighted avg       0.87      0.40      0.49     13998



## 12. Final evaluation on the untouched test set


In [32]:
y_test_proba_xgboost = (
    best_xgboost_model
    .predict_proba(X_test)[:, 1]
)

final_xgboost_test_results = (
    evaluate_predictions_from_proba(
        y_true=y_test,
        y_proba=y_test_proba_xgboost,
        threshold=(
            xgboost_selected_threshold
        ),
        model_name="Final XGBoost test"
    )
)

final_xgboost_test_results_df = pd.DataFrame([
    final_xgboost_test_results
])

final_xgboost_test_results_df


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Final XGBoost test,0.290482,0.403701,0.112229,0.816229,0.363001,0.636999,0.183771,0.197327,0.362032,...,0.151288,0.154765,4625,8116,231,1026,9142,4856,0.653093,8.910331


In [ ]:
final_xgboost_test_cm = (
    confusion_matrix_from_proba(
        y_true=y_test,
        y_proba=y_test_proba_xgboost,
        threshold=(
            xgboost_selected_threshold
        )
    )
)

final_xgboost_test_cm


,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4625,8116
Actual readmitted,231,1026


In [34]:
y_test_pred_xgboost = (
    y_test_proba_xgboost
    >= xgboost_selected_threshold
).astype(int)

print(
    classification_report(
        y_test,
        y_test_pred_xgboost,
        target_names=[
            "Not readmitted",
            "Readmitted"
        ],
        zero_division=0
    )
)


                precision    recall  f1-score   support

Not readmitted       0.95      0.36      0.53     12741
    Readmitted       0.11      0.82      0.20      1257

      accuracy                           0.40     13998
     macro avg       0.53      0.59      0.36     13998
  weighted avg       0.88      0.40      0.50     13998



In [35]:
xgboost_validation_test_comparison = (
    pd.DataFrame([
        xgboost_val_selected_results,
        final_xgboost_test_results
    ])
)

xgboost_validation_test_comparison[
    [
        "model",
        "threshold",
        "auprc",
        "auroc",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f2",
        "predicted_positive_rate",
        "patients_flagged_per_true_readmission_found"
    ]
]


,model,threshold,auprc,auroc,recall,precision,specificity,false_positive_rate,f2,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,XGBoost validation selected,0.290482,0.140877,0.628768,0.800318,0.109550,0.358214,0.641786,0.353951,0.656022,9.128231
1,Final XGBoost test,0.290482,0.151288,0.645560,0.816229,0.112229,0.363001,0.636999,0.362032,0.653093,8.910331


In [36]:
final_xgboost_test_results_df.to_csv(
    OUTPUT_DIR
    / "final_xgboost_test_metrics.csv",
    index=False
)

final_xgboost_test_cm.to_csv(
    OUTPUT_DIR
    / "final_xgboost_test_confusion_matrix.csv"
)

xgboost_validation_test_comparison.to_csv(
    OUTPUT_DIR
    / "xgboost_validation_test_comparison.csv",
    index=False
)

print("Final XGBoost results saved to:")
print(OUTPUT_DIR)


Final XGBoost results saved to:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Model_Results/xgboost_optimisation


## 13. XGBoost gain-based feature importance

The default importance type here is gain. It measures how much a feature improves the boosting objective when it is used in splits. It is useful for an initial inspection, but it does not show the direction of the relationship and it is not a causal explanation. Permutation importance or SHAP can be added later for stronger interpretation.


In [37]:
fitted_preprocessor = (
    best_xgboost_model
    .named_steps["preprocess"]
)

transformed_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

print(
    "Number of transformed features:",
    len(transformed_feature_names)
)

transformed_feature_names[:20]


Number of transformed features: 46


array(['categorical__gender_Female', 'categorical__gender_Male',
       'categorical__race_group_AfricanAmerican',
       'categorical__race_group_Caucasian',
       'categorical__race_group_Missing', 'categorical__race_group_Other',
       'categorical__age_group_30-60', 'categorical__age_group_<=30',
       'categorical__age_group_>60',
       'categorical__admission_source_group_Emergency room',
       'categorical__admission_source_group_Other',
       'categorical__admission_source_group_Physician/clinic referral',
       'categorical__discharge_group_Home',
       'categorical__discharge_group_Other',
       'categorical__medical_specialty_group_Cardiology',
       'categorical__medical_specialty_group_Family/General Practice',
       'categorical__medical_specialty_group_Internal Medicine',
       'categorical__medical_specialty_group_Missing',
       'categorical__medical_specialty_group_Other',
       'categorical__medical_specialty_group_Surgery'], dtype=object)

In [38]:
xgboost_feature_importance = (
    pd.DataFrame({
        "feature": transformed_feature_names,
        "importance": (
            fitted_xgboost
            .feature_importances_
        )
    })
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

xgboost_feature_importance.to_csv(
    OUTPUT_DIR
    / "xgboost_feature_importance_gain.csv",
    index=False
)

xgboost_feature_importance.head(30)


,feature,importance
0,categorical__discharge_group_Home,0.151581
1,numeric__number_inpatient,0.080373
2,categorical__age_group_>60,0.050893
3,categorical__diabetesMed_No,0.035748
4,categorical__primary_diagnosis_Circulatory,0.029003
5,categorical__admission_source_group_Other,0.027583
6,categorical__age_group_<=30,0.026510
7,categorical__primary_diagnosis_Respiratory,0.026289
8,categorical__admission_source_group_Emergency ...,0.026268
9,numeric__time_in_hospital,0.025741


In [39]:
def identify_original_feature(
    transformed_feature_name,
    categorical_features,
    numeric_features
):
    """Map a transformed feature back to its original variable."""

    clean_name = (
        transformed_feature_name
        .replace("categorical__", "")
        .replace("numeric__", "")
    )

    for feature in categorical_features:
        if clean_name.startswith(
            f"{feature}_"
        ):
            return feature

    for feature in numeric_features:
        if clean_name == feature:
            return feature

    return clean_name


xgboost_feature_importance[
    "original_feature"
] = (
    xgboost_feature_importance[
        "feature"
    ]
    .apply(
        identify_original_feature,
        categorical_features=categorical_features,
        numeric_features=numeric_features
    )
)

xgboost_original_feature_importance = (
    xgboost_feature_importance
    .groupby(
        "original_feature",
        as_index=False
    )["importance"]
    .sum()
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

xgboost_original_feature_importance.to_csv(
    OUTPUT_DIR
    / "xgboost_original_feature_importance_gain.csv",
    index=False
)

xgboost_original_feature_importance


,original_feature,importance
0,primary_diagnosis,0.180769
1,discharge_group,0.151581
2,age_group,0.091750
3,medical_specialty_group,0.090485
4,number_inpatient,0.080373
5,hba1c_group,0.076696
6,admission_source_group,0.069899
7,race_group,0.056943
8,diabetesMed,0.035748
9,time_in_hospital,0.025741


## 14. Save the selected settings and experiment summary


In [40]:
best_structure_parameters_table = pd.DataFrame(
    [
        {
            "parameter": parameter,
            "selected_value": str(value)
        }
        for parameter, value
        in selected_structure_params.items()
    ]
)

best_structure_parameters_table.to_csv(
    OUTPUT_DIR
    / "xgboost_best_structure_parameters.csv",
    index=False
)

selected_boosting_schedule_table = pd.DataFrame([
    {
        "selected_learning_rate": (
            selected_learning_rate
        ),
        "selected_n_estimators": (
            selected_n_estimators
        ),
        "early_stopping_rounds": (
            EARLY_STOPPING_ROUNDS
        ),
        "maximum_boosting_rounds_tested": (
            MAX_BOOSTING_ROUNDS
        )
    }
])

selected_boosting_schedule_table.to_csv(
    OUTPUT_DIR
    / "xgboost_selected_boosting_schedule.csv",
    index=False
)

selected_threshold_table = pd.DataFrame([
    {
        "recall_target": RECALL_TARGET,
        "selected_validation_threshold": (
            xgboost_selected_threshold
        )
    }
])

selected_threshold_table.to_csv(
    OUTPUT_DIR
    / "xgboost_selected_threshold.csv",
    index=False
)

experiment_summary = pd.Series({
    "structure_search_cross_validation_auprc": (
        xgboost_structure_search.best_score_
    ),
    "selected_learning_rate": (
        selected_learning_rate
    ),
    "selected_n_estimators": (
        selected_n_estimators
    ),
    "selected_validation_threshold": (
        xgboost_selected_threshold
    ),
    "validation_recall": (
        xgboost_val_selected_results[
            "recall"
        ]
    ),
    "validation_precision": (
        xgboost_val_selected_results[
            "precision"
        ]
    ),
    "validation_false_positive_rate": (
        xgboost_val_selected_results[
            "false_positive_rate"
        ]
    ),
    "test_auprc": (
        final_xgboost_test_results[
            "auprc"
        ]
    ),
    "test_auroc": (
        final_xgboost_test_results[
            "auroc"
        ]
    ),
    "test_brier_score": (
        final_xgboost_test_results[
            "brier_score"
        ]
    ),
    "test_recall": (
        final_xgboost_test_results[
            "recall"
        ]
    ),
    "test_precision": (
        final_xgboost_test_results[
            "precision"
        ]
    ),
    "test_specificity": (
        final_xgboost_test_results[
            "specificity"
        ]
    ),
    "test_false_positive_rate": (
        final_xgboost_test_results[
            "false_positive_rate"
        ]
    ),
    "test_f2": (
        final_xgboost_test_results[
            "f2"
        ]
    ),
    "test_predicted_positive_rate": (
        final_xgboost_test_results[
            "predicted_positive_rate"
        ]
    ),
    "test_patients_flagged_per_true_readmission": (
        final_xgboost_test_results[
            "patients_flagged_per_true_readmission_found"
        ]
    )
})

experiment_summary.to_csv(
    OUTPUT_DIR
    / "xgboost_experiment_summary.csv",
    header=["value"]
)

experiment_summary


structure_search_cross_validation_auprc         0.150838
selected_learning_rate                          0.100000
selected_n_estimators                         196.000000
selected_validation_threshold                   0.290482
validation_recall                               0.800318
validation_precision                            0.109550
validation_false_positive_rate                  0.641786
test_auprc                                      0.151288
test_auroc                                      0.645560
test_brier_score                                0.154765
test_recall                                     0.816229
test_precision                                  0.112229
test_specificity                                0.363001
test_false_positive_rate                        0.636999
test_f2                                         0.362032
test_predicted_positive_rate                    0.653093
test_patients_flagged_per_true_readmission      8.910331
dtype: float64

## 15. Main outputs for later model comparison

The most important files are:

- `final_xgboost_test_metrics.csv`
- `final_xgboost_test_confusion_matrix.csv`
- `xgboost_validation_test_comparison.csv`
- `xgboost_experiment_summary.csv`
- `xgboost_original_feature_importance_gain.csv`

For the final comparison, place most weight on test AUPRC, AUROC, recall, false-positive rate, precision, predicted-positive rate, and patients flagged per true readmission found. Accuracy alone is not suitable for this imbalanced outcome.


The tuned XGBoost model achieved performance very similar to the tuned Random Forest. At a validation-selected threshold targeting at least 80% recall, XGBoost obtained 81.6% recall, 11.2% precision and a false-positive rate of 63.7% on the test set. It slightly improved threshold-based measures compared with Random Forest and more clearly improved over the single Decision Tree, but the differences were modest. The substantial overlap between predicted scores for readmitted and non-readmitted patients suggests that the available variables provide limited discrimination. Strong regularisation and shallow trees were selected during tuning, indicating that greater model complexity did not produce better generalisation. Additionally, class weighting caused poor probability calibration, meaning that the model scores should not be interpreted directly as absolute readmission probabilities.